In [ ]:
%run "./00_CC_Mapping_Setup.ipynb"

In [ ]:
%sql
/* ===================================================================================
   METRIC: EBA07 - Marketing
   
   BUSINESS QUESTION: 
   During the assessment period, did the Unit engage in marketing activities or 
   commercial sponsorships (including but not limited to expert sponsorships?)
   
   LOGIC SUMMARY:
   1. FINANCE DATA ONLY: Unions the 6 Finance files. Coupa data is excluded.
   2. CATEGORY CODE FILTER: Strictly filters the Finance data for the predefined 
      Sponsorship/Marketing Category Codes (194, 370, 372, 377, 408, 628, 629, 653, 
      830, 875, 877).
   3. MAPPING & OUTPUT: Joins the standardized Cost Center Mapping Bootstrap to the 
      flagged transactions using safe Integer casting. Rolls up by AU_ID. If any 
      marketing transaction exists for the Assessable Unit, outputs 'Yes'. Otherwise, 'No'.
=================================================================================== */

WITH Combined_Finance_Raw AS (
    -- 1. Union all 6 Finance files into one master dataset
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Marketing_Transactions AS (
    -- 2. Clean the Finance columns and filter STRICTLY for marketing category codes
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        TRIM(CatCode) AS CatCode
    FROM Combined_Finance_Raw
    WHERE TRIM(CatCode) IN (
        '194', '370', '372', '377', '408', '628', 
        '629', '653', '830', '875', '877'
    )
),

Mapped_AUs AS (
    -- 3a. Join the flagged transactions to the standardized CC Mapping
    SELECT 
        m.AU_ID AS BusinessID,
        m.AU_Name,
        CASE WHEN mt.Cleaned_CC IS NOT NULL THEN 1 ELSE 0 END AS Has_Marketing
    FROM vw_cost_center_mapping_bootstrap m
    
    -- LEFT JOIN so all AUs appear in the final output
    LEFT JOIN Marketing_Transactions mt
        ON CAST(m.Cost_Center_ID AS INT) = mt.Cleaned_CC
)

-- 3b. Roll everything up to match the Final Output Template
SELECT 
    BusinessID,                          
    AU_Name,                             
    'EBA07' AS QuestionID,               
    CASE 
        WHEN SUM(Has_Marketing) > 0 THEN 'Yes' 
        ELSE 'No' 
    END AS Response
    
FROM Mapped_AUs
GROUP BY BusinessID, AU_Name
ORDER BY BusinessID;

In [ ]:
%sql
/* ===================================================================================
   DEBUG TABLE: EBA07 - Marketing Drill-Down
   
   PURPOSE: 
   Provides a row-by-row view of every Cost Center transaction that triggered a 'Yes' 
   for EBA07. Verifies that the matched Category Code belongs to the marketing list.
=================================================================================== */

WITH Combined_Finance_Raw AS (
    SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_1
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_2
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_3
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_4
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_5
    UNION ALL SELECT CostCenter, CatCode FROM hive_metastore.ra_adido_2025.f25_finance_data_6
),

Marketing_Transactions AS (
    SELECT 
        TRY_CAST(TRIM(CostCenter) AS INT) AS Cleaned_CC,
        TRIM(CatCode) AS CatCode,
        CostCenter AS Raw_Cost_Center_String
    FROM Combined_Finance_Raw
    WHERE TRIM(CatCode) IN (
        '194', '370', '372', '377', '408', '628', 
        '629', '653', '830', '875', '877'
    )
)

SELECT DISTINCT
    m.AU_ID AS BusinessID,
    m.AU_Name,
    m.Cost_Center_ID AS Mapped_Cost_Center,
    mt.Raw_Cost_Center_String,
    mt.CatCode AS Extracted_Category_Code,
    CASE 
        WHEN mt.CatCode = '370' THEN 'Charitable Sponsorships'
        WHEN mt.CatCode = '372' THEN 'C&PA Sponsorship-Marketing only'
        WHEN mt.CatCode = '628' THEN 'Sponsorship Fees'
        ELSE 'Other Marketing Code'
    END AS Category_Description
    
FROM vw_cost_center_mapping_bootstrap m

-- INNER JOIN to only return AUs that successfully mapped to a marketing transaction
INNER JOIN Marketing_Transactions mt
    ON CAST(m.Cost_Center_ID AS INT) = mt.Cleaned_CC
    
ORDER BY BusinessID;